# Task 4 - Distributed Computing & Imbalance (Week 4)
**Notebook:** `notebooks/Task4.ipynb`
**Aligned lectures:** Performance Tuning & Debugging; Handling Imbalanced Data; NLP/Time Series.

Spark UI interpretation, caching/persistence, resource configuration, repartitioning, and **handling class imbalance** (positive reviews dominate).

In [1]:
# === Colab bootstrap: install Spark (run once per session) ===
!pip -q install pyspark==3.5.1
import os, time
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("7006SCN_AmazonReviews_Sentiment")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "12g")
         .config("spark.driver.maxResultSize", "2g")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 20.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
Spark version: 3.5.1
Spark UI: http://a6d15af2ead6:4040


In [2]:
# Mount Google Drive so data/models persist across the six notebooks.
from google.colab import drive
drive.mount('/content/drive')
BASE   = '/content/drive/MyDrive/7006SCN'
RAW    = BASE + '/raw'                      # downloaded jsonl.gz
PARQ   = BASE + '/reviews_parquet'          # raw -> parquet
PROC   = BASE + '/reviews_processed'        # after Task2 NLP pipeline
EXPORT = BASE + '/tableau'                  # Task6 exports
import os
for p in (BASE, RAW, EXPORT):
    os.makedirs(p, exist_ok=True)
print('Base:', BASE)

Mounted at /content/drive
Base: /content/drive/MyDrive/7006SCN


## 4.1 Expose the Spark UI in Colab (screenshot Jobs/Stages/Storage)

In [3]:
from google.colab.output import eval_js
print("Spark UI:", eval_js("google.colab.kernel.proxyPort(4040)"))
print("Local:", spark.sparkContext.uiWebUrl)

Spark UI: https://4040-gpu-t4-s-kkb-usw4a1-muo6qc08f95y-a.us-west4-1.prod.colab.dev
Local: http://a6d15af2ead6:4040


## 4.2 Caching / persisting - justified
The TF-IDF train set is reused by every model and CV fold; without caching Spark re-reads Parquet and recomputes each time. We persist and compare timings.

In [4]:
import time
from pyspark import StorageLevel
train=spark.read.parquet(PROC+"_train")
train.unpersist()
t0=time.time(); train.count(); train.groupBy("label").count().collect(); uncached=time.time()-t0
train.persist(StorageLevel.MEMORY_AND_DISK); train.count()
t0=time.time(); train.count(); train.groupBy("label").count().collect(); cached=time.time()-t0
print(f"Uncached: {uncached:.2f}s | Cached: {cached:.2f}s | Speed-up: {uncached/max(cached,1e-9):.1f}x")

Uncached: 43.62s | Cached: 11.64s | Speed-up: 3.7x


## 4.3 Resource configuration (screenshot)

In [5]:
for k,v in sorted(spark.sparkContext.getConf().getAll()):
    if any(t in k for t in ("memory","cores","partitions","executor","driver","parallelism")):
        print(f"{k} = {v}")
print("Default parallelism:", spark.sparkContext.defaultParallelism)

spark.driver.extraJavaOptions = -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false
spark.driver.host = a6d15af2ead6
spark.driver.maxResultSize = 2g
spark.driver.memory = 12g
spark.driver.port = 4

## 4.4 Repartitioning (before/after timing)

In [6]:
print("Partitions before:", train.rdd.getNumPartitions())
t0=time.time(); train.groupBy("label").count().collect(); before=time.time()-t0
train_rp=train.repartition(64)
print("Partitions after:", train_rp.rdd.getNumPartitions())
t0=time.time(); train_rp.groupBy("label").count().collect(); after=time.time()-t0
print(f"before {before:.2f}s | after {after:.2f}s")

Partitions before: 11
Partitions after: 64
before 6.88s | after 14.12s


## 4.5 Handling class imbalance
Positive reviews far outnumber negative, so accuracy is misleading and the model can ignore the minority. We add per-row **class weights** (inverse class frequency); Logistic Regression and LinearSVC accept a `weightCol`. Report the weighted model's recall on the negative class vs the unweighted baseline.

In [7]:
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Sample so the JVM doesn't OOM (still demonstrates imbalance handling)
samp = train.sample(False, 0.10, seed=42).cache()
n = samp.count()
pos = samp.filter(F.col("label")==1).count(); neg = n - pos
w_pos = n/(2.0*pos); w_neg = n/(2.0*neg)
samp_w = samp.withColumn("w", F.when(F.col("label")==1, w_pos).otherwise(w_neg))
print(f"sample n={n:,} pos={pos:,} neg={neg:,} | w_pos={w_pos:.3f} w_neg={w_neg:.3f}")

test = spark.read.parquet(PROC+"_test")
rec = MulticlassClassificationEvaluator(labelCol="label",
        metricName="recallByLabel", metricLabel=0.0)   # recall on negative class

# Fit ONE at a time
base = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20).fit(samp)
print("baseline negative-class recall:", round(rec.evaluate(base.transform(test)),4))

wgt = LogisticRegression(featuresCol="features", labelCol="label",
                         weightCol="w", maxIter=20).fit(samp_w)
print("weighted negative-class recall:", round(rec.evaluate(wgt.transform(test)),4))

sample n=1,504,199 pos=1,184,187 neg=320,012 | w_pos=0.635 w_neg=2.350
baseline negative-class recall: 0.7596
weighted negative-class recall: 0.8942


## 4.6 Interpretation (write-up)
The Spark UI DAG shows the TF-IDF transform and the `groupBy` shuffle stage; caching removes the repeated Parquet scan (visible in the Storage tab once persisted). Repartitioning evens task durations. Class weighting trades a little overall accuracy for a large gain in negative-class recall - essential when the business cares about catching dissatisfied customers. The same code scales unchanged on a multi-node cluster.

## Version control
>=3 commits: `Task4: Spark UI + cache timing`, `Task4: repartition + resource config`, `Task4: class-weight imbalance handling`.